# MTAH 2027 — fine-tune a ball detector (Jalur 1)

Fine-tunes a COCO-pretrained YOLO on 640×640 tiles of the BAR-ATH tactical video, with boxes taken
from the **ATD's own ball** (automatic labels, projected to pixels with the provided homography — no
human labels). Tiles are at native resolution, so the detector later runs on full 1920×1080 frames at
`imgsz=1920` with the same pixel scale.

Train and val tiles come from disjoint time blocks (every 10th minute of video is val). The decision
that counts (Stage A in WORKLOG.md) is measured locally afterwards with
`python -m mtah.vball probe --split val --weights <best.pt>`, not with the mAP printed here.

Before the first run:
1. Runtime → Change runtime type → **A100 GPU**.
2. Build the dataset locally (`python -m mtah.vball dataset --out outputs/vball/dataset_v2 --n_train 15000 --n_val 1500`,
   then `python -m mtah.vball refine --dataset outputs/vball/dataset_v2 --drop_unsnapped`), zip it
   (`outputs/vball/vball_dataset_v2.zip`) and upload it to Drive at `DRIVE_ROOT/vball/`.


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/Metrica-Hackathon/mtah"
ZIP = f"{DRIVE_ROOT}/vball/vball_dataset_v2.zip"
RUN = "ft1"
MODEL = "yolo11s.pt"        # yolo11n.pt is ~2x faster at inference, a bit less accurate
EPOCHS = 60


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics==8.4.212


In [ ]:
# dataset to local disk (much faster than reading from Drive)
!rm -rf /content/ds && mkdir -p /content/ds && cp "{ZIP}" /content/ds.zip && unzip -q /content/ds.zip -d /content/ds
!ls /content/ds/dataset_v2/images/train | wc -l; ls /content/ds/dataset_v2/images/val | wc -l
open('/content/ds/dataset_v2/data.yaml', 'w').write(
    "path: /content/ds/dataset_v2\ntrain: images/train\nval: images/val\nnames:\n  0: ball\n")


Training. Small `scale` augmentation on purpose: the ball's pixel size must stay close to what the
detector sees on full frames. Re-running after a disconnect: set `resume=True` and `model` to
`{OUT}/{RUN}/weights/last.pt`.

In [ ]:
from ultralytics import YOLO
OUT = f"{DRIVE_ROOT}/vball/runs"
model = YOLO(MODEL)
model.train(data="/content/ds/dataset_v2/data.yaml", imgsz=640, epochs=EPOCHS, batch=64, workers=8,
            project=OUT, name=RUN, exist_ok=True, seed=0, deterministic=True, patience=20,
            scale=0.2, translate=0.1, degrees=0.0, mosaic=1.0, close_mosaic=10, fliplr=0.5,
            cache="ram", plots=True)


In [ ]:
import glob
print(glob.glob(f"{OUT}/{RUN}/weights/*.pt"))
!tail -5 "{OUT}/{RUN}/results.csv"


Download `vball/runs/<RUN>/weights/best.pt` from Drive to the laptop as
`outputs/vball/<RUN>/best.pt`, then locally:

```bash
python -m mtah.vball probe --split val --weights outputs/vball/ft1/best.pt --imgsz 1920 --device mps
```
